# 04_model_improvements



# Model Optimization

In the previous notebook, XGBoost performed better than Logistic Regression,
Decision Trees, Random Forest and stacking. Adding `purpose` and `int_rate`
also improved its performance.

Here, we compare XGBoost with LightGBM, CatBoost and HistGradientBoosting
using the same loan information. We then tune the promising models and
select one for final evaluation.

PR-AUC is our primary metric because defaults are the minority class.
We will also compare ROC-AUC, F1, recall and training time.

The chronological split remains unchanged. We will tune using data from
the training period, compare models on late-2015 validation data and
reserve 2016 for final testing.


In [1]:

from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import polars as pl

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

source = root / "data/interim/lendingclub_2013_2016_36m_raw.parquet"
manifest = pd.read_csv(
    root / "reports/preprocessing/02_feature_manifest_draft.csv"
)

numeric_features = manifest.loc[
    manifest["model_status"] == "CANDIDATE_NUMERIC", "feature"
].tolist()

categorical_features = ["home_ownership", "addr_state", "purpose"]

columns = list(dict.fromkeys(
    numeric_features
    + categorical_features
    + ["emp_length", "int_rate", "issue_date", "target"]
))

lf = pl.scan_parquet(source)

train_lf = lf.filter(
    (pl.col("issue_date") >= date(2013, 1, 1))
    & (pl.col("issue_date") < date(2015, 10, 1))
)

val_lf = lf.filter(
    (pl.col("issue_date") >= date(2015, 10, 1))
    & (pl.col("issue_date") < date(2016, 1, 1))
)

train_df = train_lf.select(columns).collect().to_pandas()
val_df = val_lf.select(columns).collect().to_pandas()

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Default rates:")
print("Train:", round(train_df["target"].mean(), 4))
print("Validation:", round(val_df["target"].mean(), 4))


Training: (457355, 57)
Validation: (88810, 57)
Default rates:
Train: 0.1393
Validation: 0.1484


In [2]:

# Update the selected feature manifest

feature_config = manifest.copy()

numeric = set(
    feature_config.loc[
        feature_config["model_status"] == "CANDIDATE_NUMERIC",
        "feature"
    ]
)

categorical = {"home_ownership", "addr_state", "purpose"}
applicant = numeric | categorical | {"emp_length"}
origination = applicant | {"int_rate"}

# Add any features missing from the original manifest
for feature in ["purpose", "int_rate"]:
    if feature not in feature_config["feature"].values:
        feature_config = pd.concat([
            feature_config,
            pd.DataFrame({"feature": [feature]})
        ], ignore_index=True)

feature_config.loc[
    feature_config["feature"] == "purpose", "model_status"
] = "APPROVED_CATEGORICAL"

feature_config.loc[
    feature_config["feature"] == "int_rate", "model_status"
] = "APPROVED_NUMERIC_ORIGINATION"

feature_config["applicant_model"] = (
    feature_config["feature"].isin(applicant)
)

feature_config["origination_model"] = (
    feature_config["feature"].isin(origination)
)

config_dir = root / "config"
config_dir.mkdir(exist_ok=True)

feature_config.to_csv(
    config_dir / "feature_manifest.csv",
    index=False
)

print("Applicant features:", feature_config["applicant_model"].sum())
print("Origination features:", feature_config["origination_model"].sum())
print("Saved: config/feature_manifest.csv")


Applicant features: 54
Origination features: 55
Saved: config/feature_manifest.csv



### Quick prediction experiment

Our full model uses more information than a user can reasonably enter
manually. To test a simpler interface, we keep 15 borrower and loan
features and replace the remaining inputs with historical values
calculated from the training set.

We compare this approximation with the full-input predictions using
the same fitted XGBoost model and the 2015 validation set.

This will help us decide whether Quick Prediction can use the full
model or whether a separate reduced-feature model is necessary.


In [4]:

import cloudpickle
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)

model_path = root / "saved_models/03_origination_xgb_candidate.pkl"

with open(model_path, "rb") as f:
    bundle = cloudpickle.load(f)

preprocessor = bundle["preprocessor"]
model = bundle["model"]
threshold = float(bundle["threshold"])
model_features = bundle["input_columns"]

manual_features = [
    "loan_amnt",
    "int_rate",
    "purpose",
    "annual_inc",
    "emp_length_years",
    "home_ownership",
    "addr_state",
    "fico_range_low",
    "fico_range_high",
    "dti",
    "revol_util",
    "open_acc",
    "total_acc",
    "delinq_2yrs",
    "inq_last_6mths",
]

assert len(manual_features) == 15
missing_features = sorted(set(manual_features) - set(model_features))

print("Proposed GUI inputs:", len(manual_features))
print("Features absent from trained model:", missing_features)

if missing_features:
    print("\nAvailable model inputs:")
    print(sorted(model_features))

# Use the same purpose grouping established in Notebook 03.
train_purpose = (
    train_df["purpose"].astype("string")
    .str.strip().str.lower()
    .replace("", pd.NA).fillna("missing")
)

common_purposes = set(
    train_purpose.value_counts()
    .loc[lambda counts: counts >= 1000].index
)


def prepare_inputs(df):
    X = pd.DataFrame(index=df.index)

    for col in numeric_features:
        values = df[col].astype("string").str.strip()

        if col == "revol_util":
            values = values.str.replace("%", "", regex=False)

        X[col] = pd.to_numeric(
            values, errors="coerce"
        ).astype(np.float32)

        if col == "dti":
            X[col] = X[col].where(X[col] >= 0)

    emp = df["emp_length"].astype("string").str.strip()

    years = pd.to_numeric(
        emp.str.extract(r"^(\d+)\s+years?$", expand=False),
        errors="coerce",
    )
    years = years.mask(emp == "< 1 year", 0)
    years = years.mask(emp == "10+ years", 10)

    X["emp_length_years"] = years.astype(np.float32)

    X["delinq_recency_missing"] = (
        X["mths_since_last_delinq"].isna().astype(np.int8)
    )
    X["emp_length_missing"] = (
        X["emp_length_years"].isna().astype(np.int8)
    )
    X["recent_inq_missing"] = (
        X["mths_since_recent_inq"].isna().astype(np.int8)
    )

    for col in ["home_ownership", "addr_state"]:
        X[col] = df[col].astype("string").str.strip().astype(object)

    purpose = (
        df["purpose"].astype("string")
        .str.strip().str.lower()
        .replace("", pd.NA).fillna("missing")
    )

    X["purpose"] = (
        purpose.where(purpose.isin(common_purposes), "rare_other")
        .astype(object)
    )

    X["int_rate"] = pd.to_numeric(
        df["int_rate"].astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce",
    ).astype(np.float32)

    return X[model_features].copy()


X_train_full = prepare_inputs(train_df)
X_val_full = prepare_inputs(val_df)

# Replace unprovided features with training-only statistics.
X_val_quick = X_val_full.copy()
historical_defaults = {}

for col in model_features:
    if col in manual_features:
        continue

    values = X_train_full[col]

    if pd.api.types.is_numeric_dtype(values):
        default = values.median()
    else:
        default = values.mode(dropna=True).iloc[0]

    historical_defaults[col] = default
    X_val_quick[col] = default

# Compare predictions from the same fitted model.
full_prob = model.predict_proba(
    preprocessor.transform(X_val_full).astype(np.float32)
)[:, 1]

quick_prob = model.predict_proba(
    preprocessor.transform(X_val_quick).astype(np.float32)
)[:, 1]

y_true = val_df["target"].to_numpy()

def evaluate(name, probabilities):
    pred = (probabilities >= threshold).astype(int)

    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "f1": f1_score(y_true, pred),
        "precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "recall": recall_score(y_true, pred),
        "brier": brier_score_loss(y_true, probabilities),
    }


quick_results = pd.DataFrame([
    evaluate("Full features", full_prob),
    evaluate("15 + historical defaults", quick_prob),
])

display(quick_results.round(4))

print("Manual inputs:", len(manual_features))
print("Historically filled:", len(historical_defaults))
print("Classification threshold:", round(threshold, 4))
print("2016 test set untouched.")


Proposed GUI inputs: 15
Features absent from trained model: ['fico_range_high']

Available model inputs:
['acc_now_delinq', 'acc_open_past_24mths', 'addr_state', 'annual_inc', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'delinq_2yrs', 'delinq_amnt', 'delinq_recency_missing', 'dti', 'emp_length_missing', 'emp_length_years', 'fico_range_low', 'home_ownership', 'inq_last_6mths', 'int_rate', 'loan_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_last_delinq', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'open_acc', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec', 'pub_rec_bankruptcies', 'purpose', 'recent_inq_missing', 'revol_bal', 'revol_util', 

,model,roc_auc,pr_auc,f1,precision,recall,brier
0,Full features,0.7136,0.2986,0.3625,0.2692,0.5547,0.1176
1,15 + historical defaults,0.6906,0.2706,0.3367,0.2640,0.4645,0.1206


Manual inputs: 15
Historically filled: 44
Classification threshold: 0.1501
2016 test set untouched.


In [5]:

# Compare the corrected 15-input Quick mode

manual_features = [
    f for f in manual_features
    if f != "fico_range_high"
] + ["revol_bal"]

assert len(manual_features) == 15
assert set(manual_features).issubset(model_features)

X_val_quick_15 = X_val_full.copy()
historical_defaults_15 = {}

for col in model_features:
    if col in manual_features:
        continue

    values = X_train_full[col]

    if pd.api.types.is_numeric_dtype(values):
        default = values.median()
    else:
        default = values.mode(dropna=True).iloc[0]

    historical_defaults_15[col] = default
    X_val_quick_15[col] = default

quick_prob_15 = model.predict_proba(
    preprocessor.transform(
        X_val_quick_15
    ).astype(np.float32)
)[:, 1]

quick_results_15 = pd.DataFrame([
    evaluate("Full features", full_prob),
    evaluate("14 inputs + historical", quick_prob),
    evaluate("15 inputs + historical", quick_prob_15),
])

display(quick_results_15.round(4))

print("Manual features:", len(manual_features))
print("Historically filled:", len(historical_defaults_15))
print("Threshold:", threshold)


,model,roc_auc,pr_auc,f1,precision,recall,brier
0,Full features,0.7136,0.2986,0.3625,0.2692,0.5547,0.1176
1,14 inputs + historical,0.6906,0.2706,0.3367,0.2640,0.4645,0.1206
2,15 inputs + historical,0.6921,0.2728,0.3384,0.2660,0.4649,0.1205


Manual features: 15
Historically filled: 43
Threshold: 0.15012823045253754


above Two experiments tell us that limiting our input feature to just 15 and filling rest of them historically does affect the result , but it is minor tradeoff to be considered , we will also have option to input all features to give our best score. 


## 04.2 Feature preparation

We use the selected features from the preprocessing audit, along
with `purpose` and `int_rate`, which improved XGBoost performance
in the previous experiments.

Numerical columns are converted to a consistent format, employment
length is converted to years, and three missing-value indicators
are retained. Rare loan purposes are grouped using training data.

We keep the original chronological split. Imputation and encoding
will be fitted separately on the training data in the next step.


In [3]:

# Prepare the selected features for model comparison

numeric_cols = numeric_features + ["emp_length_years", "int_rate"]

indicator_cols = [
    "delinq_recency_missing",
    "emp_length_missing",
    "recent_inq_missing",
]

categorical_cols = ["home_ownership", "addr_state", "purpose"]

model_cols = numeric_cols + indicator_cols + categorical_cols


def clean_purpose(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace("", pd.NA)
        .fillna("missing")
    )


purpose_counts = clean_purpose(train_df["purpose"]).value_counts()
common_purposes = set(purpose_counts[purpose_counts >= 1000].index)


def prepare_features(df):
    X = pd.DataFrame(index=df.index)

    for col in numeric_features:
        values = df[col].astype("string").str.strip()

        if col == "revol_util":
            values = values.str.replace("%", "", regex=False)

        X[col] = pd.to_numeric(
            values, errors="coerce"
        ).astype(np.float32)

    X["dti"] = X["dti"].where(
        np.isfinite(X["dti"]) & (X["dti"] >= 0)
    )

    emp = df["emp_length"].astype("string").str.strip()

    years = pd.to_numeric(
        emp.str.extract(r"^(\d+)\s+years?$", expand=False),
        errors="coerce",
    )
    years = years.mask(emp == "< 1 year", 0)
    years = years.mask(emp == "10+ years", 10)

    X["emp_length_years"] = years.astype(np.float32)

    X["int_rate"] = pd.to_numeric(
        df["int_rate"].astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce",
    ).astype(np.float32)

    X["delinq_recency_missing"] = (
        X["mths_since_last_delinq"].isna().astype(np.int8)
    )
    X["emp_length_missing"] = (
        X["emp_length_years"].isna().astype(np.int8)
    )
    X["recent_inq_missing"] = (
        X["mths_since_recent_inq"].isna().astype(np.int8)
    )

    for col in ["home_ownership", "addr_state"]:
        X[col] = (
            df[col].astype("string").str.strip()
            .fillna("MISSING").astype(object)
        )

    purpose = clean_purpose(df["purpose"])
    X["purpose"] = (
        purpose.where(purpose.isin(common_purposes), "rare_other")
        .astype(object)
    )

    return X[model_cols]


X_train = prepare_features(train_df)
X_val = prepare_features(val_df)

y_train = train_df["target"].to_numpy(dtype=np.int8)
y_val = val_df["target"].to_numpy(dtype=np.int8)

train_dates = pd.to_datetime(train_df["issue_date"]).reset_index(drop=True)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Default rate:", round(y_train.mean(), 4))
print("Categorical features:", categorical_cols)

print("\nMissing values in training:")
display(
    X_train.isna().mean()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
    .head(10)
    .round(3)
)


Train: (457355, 58)
Validation: (88810, 58)
Default rate: 0.1393
Categorical features: ['home_ownership', 'addr_state', 'purpose']

Missing values in training:


mths_since_last_delinq    0.502
mths_since_recent_inq     0.105
emp_length_years          0.059
mo_sin_old_il_acct        0.038
num_tl_120dpd_2m          0.034
bc_util                   0.010
percent_bc_gt_75          0.010
bc_open_to_buy            0.010
mths_since_recent_bc      0.009
revol_util                0.000
dtype: float64


## 04.3 Preprocessing for model comparison

All models will use the same selected borrower and loan information.

For XGBoost, LightGBM and HistGradientBoosting, we use median
imputation for numerical features and one-hot encoding for categorical
features.

CatBoost can handle categorical variables directly, so we retain their
original categories rather than one-hot encoding them. The numerical
imputation remains the same.

We do not standardize the features because these are tree-based models.
All preprocessing is fitted using training data only.


In [4]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_inputs = numeric_cols + indicator_cols

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_inputs,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="constant",
                    fill_value="MISSING"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32
                )),
            ]),
            categorical_cols,
        ),
    ],
    sparse_threshold=0,
)

# Encoded data for XGBoost, LightGBM and HistGB
X_train_encoded = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_encoded = preprocessor.transform(X_val).astype(np.float32)

# CatBoost keeps categorical columns in their original form
numeric_imputer = preprocessor.named_transformers_["numeric"]

def prepare_catboost(X):
    result = X.copy()

    result[numeric_inputs] = numeric_imputer.transform(
        X[numeric_inputs]
    ).astype(np.float32)

    for col in categorical_cols:
        result[col] = result[col].fillna("MISSING").astype(str)

    return result

X_train_cat = prepare_catboost(X_train)
X_val_cat = prepare_catboost(X_val)

print("Encoded training:", X_train_encoded.shape)
print("Encoded validation:", X_val_encoded.shape)

print("\nCatBoost training:", X_train_cat.shape)
print("Categorical columns:", categorical_cols)

print("\nMissing values after preprocessing:")
print("Encoded:", np.isnan(X_train_encoded).sum())
print("CatBoost numeric:", X_train_cat[numeric_inputs].isna().sum().sum())


Encoded training: (457355, 122)
Encoded validation: (88810, 122)

CatBoost training: (457355, 58)
Categorical columns: ['home_ownership', 'addr_state', 'purpose']

Missing values after preprocessing:
Encoded: 0
CatBoost numeric: 0



## Comparing gradient boosting models

We compare XGBoost, LightGBM, CatBoost and HistGradientBoosting
using the same training and validation periods.

The first comparison uses reasonable fixed hyperparameters rather
than tuning each model immediately. XGBoost, LightGBM and HistGB
use one-hot encoded features, while CatBoost uses the categorical
variables directly.

We mainly compare PR-AUC and ROC-AUC, along with F1-score,
precision, recall and training time. The strongest models will
be considered for further tuning.


Xgboost

In [8]:

import time
from xgboost import XGBClassifier

family_runs = {}

xgb_config = {
    "n_estimators": 350,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 10,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5,
    "objective": "binary:logistic",
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "random_state": 42,
    "n_jobs": 4,
}

xgb = XGBClassifier(**xgb_config)

start = time.perf_counter()
xgb.fit(X_train_encoded, y_train)

family_runs["XGBoost"] = {
    "model": xgb,
    "prob": xgb.predict_proba(X_val_encoded)[:, 1],
    "seconds": time.perf_counter() - start,
    "config": xgb_config,
}

print(f"XGBoost: {family_runs['XGBoost']['seconds']:.1f}s")


XGBoost: 9.1s


catboost


In [9]:

from catboost import CatBoostClassifier

cat_config = {
    "iterations": 350,
    "depth": 5,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5,
    "loss_function": "Logloss",
    "random_seed": 42,
    "thread_count": 4,
    "verbose": False,
    "allow_writing_files": False,
}

cat = CatBoostClassifier(**cat_config)

start = time.perf_counter()

cat.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_cols,
)

family_runs["CatBoost"] = {
    "model": cat,
    "prob": cat.predict_proba(X_val_cat)[:, 1],
    "seconds": time.perf_counter() - start,
    "config": cat_config,
}

print(f"CatBoost: {family_runs['CatBoost']['seconds']:.1f}s")


CatBoost: 67.5s


HistGradientBoosting

In [10]:

from sklearn.ensemble import HistGradientBoostingClassifier

hist_config = {
    "max_iter": 350,
    "max_depth": 5,
    "max_leaf_nodes": 31,
    "learning_rate": 0.05,
    "min_samples_leaf": 50,
    "l2_regularization": 5,
    "early_stopping": False,
    "random_state": 42,
}

hist = HistGradientBoostingClassifier(**hist_config)

start = time.perf_counter()
hist.fit(X_train_encoded, y_train)

family_runs["HistGradientBoosting"] = {
    "model": hist,
    "prob": hist.predict_proba(X_val_encoded)[:, 1],
    "seconds": time.perf_counter() - start,
    "config": hist_config,
}

print(
    "HistGradientBoosting:",
    f"{family_runs['HistGradientBoosting']['seconds']:.1f}s"
)


HistGradientBoosting: 24.4s


comparision 


In [11]:

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    brier_score_loss,
)

models_to_compare = [
    "XGBoost",
    "CatBoost",
    "HistGradientBoosting",
]


def evaluate_model(name, run):
    prob = np.asarray(run["prob"]).ravel()

    precision, recall, thresholds = precision_recall_curve(
        y_val, prob
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )

    idx = int(np.argmax(f1_values))
    threshold = float(thresholds[idx])

    pred = (prob >= threshold).astype(int)
    pred_05 = (prob >= 0.5).astype(int)

    return {
        "model": name,
        "roc_auc": roc_auc_score(y_val, prob),
        "pr_auc": average_precision_score(y_val, prob),
        "f1_at_0.5": f1_score(y_val, pred_05),
        "best_f1": f1_score(y_val, pred),
        "threshold": threshold,
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "balanced_acc": balanced_accuracy_score(y_val, pred),
        "brier": brier_score_loss(y_val, prob),
        "train_seconds": run["seconds"],
    }


completed = [
    name for name in models_to_compare
    if name in family_runs
]

family_results = pd.DataFrame([
    evaluate_model(name, family_runs[name])
    for name in completed
])

family_results = (
    family_results
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(family_results.round(4))

missing = set(models_to_compare) - set(completed)

if missing:
    print("Not yet evaluated:", ", ".join(sorted(missing)))

report_dir = root / "reports/modeling"
report_dir.mkdir(parents=True, exist_ok=True)

family_results.to_csv(
    report_dir / "04_boosting_family_comparison.csv",
    index=False
)

print("\nBest initial model:", family_results.iloc[0]["model"])
print("2016 test set untouched.")


,model,roc_auc,pr_auc,f1_at_0.5,best_f1,threshold,precision,recall,balanced_acc,brier,train_seconds
0,XGBoost,0.7133,0.2980,0.0074,0.3633,0.1501,0.2645,0.5799,0.6494,0.1174,9.0753
1,HistGradientBoosting,0.7136,0.2972,0.0083,0.3641,0.1520,0.2656,0.5788,0.6499,0.1173,24.4086
2,CatBoost,0.7128,0.2963,0.0053,0.3630,0.1566,0.2723,0.5441,0.6453,0.1176,67.5447



Best initial model: XGBoost
2016 test set untouched.



## Hyperparameter tuning

The initial boosting models achieved similar results, so we now
investigate whether tuning can improve their performance.

We use an internal chronological split from the training period.
The main validation set is kept separate for the final comparison.

We select configurations primarily using PR-AUC.



## 04.5 XGBoost hyperparameter tuning

We use a small grid search around the initial XGBoost configuration.
The search varies tree depth, number of estimators and learning rate.

To maintain chronological evaluation, models are fitted on loans
issued before April 2015 and evaluated on April–September 2015.
PR-AUC is used to select the best configuration.

The late-2015 validation set remains separate for comparing the
selected models.


In [12]:

from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from xgboost import XGBClassifier
from IPython.display import display

train_dates = pd.to_datetime(
    train_df["issue_date"]
).reset_index(drop=True)

# Earlier loans for fitting, later loans for tuning
fold = np.where(
    train_dates < pd.Timestamp("2015-04-01"),
    -1,
    0,
)

cv = PredefinedSplit(test_fold=fold)

xgb_base = {
    "learning_rate": 0.05,
    "min_child_weight": 10,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5,
    "objective": "binary:logistic",
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "random_state": 42,
    "n_jobs": 4,
}

xgb_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", XGBClassifier(**xgb_base)),
])

param_grid = {
    "model__max_depth": [3, 4],
    "model__n_estimators": [350, 600],
    "model__learning_rate": [0.03, 0.05],
}

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    refit=False,
    n_jobs=1,
    verbose=0,
)

xgb_search.fit(X_train, y_train)

xgb_grid_results = pd.DataFrame(
    xgb_search.cv_results_
)[
    [
        "param_model__max_depth",
        "param_model__n_estimators",
        "param_model__learning_rate",
        "mean_test_score",
        "mean_fit_time",
    ]
].sort_values("mean_test_score", ascending=False)

display(xgb_grid_results.round(4))

xgb_best_config = {
    **xgb_base,
    **{
        key.replace("model__", ""): value
        for key, value in xgb_search.best_params_.items()
    },
}

print("\nBest internal PR-AUC:", round(xgb_search.best_score_, 4))
print("Best XGBoost configuration:", xgb_best_config)


,param_model__max_depth,param_model__n_estimators,param_model__learning_rate,mean_test_score,mean_fit_time
7,4,600,0.05,0.2925,12.8766
3,4,600,0.03,0.2916,11.9993
5,3,600,0.05,0.2916,9.8383
6,4,350,0.05,0.2910,7.6670
4,3,350,0.05,0.2892,6.9118
1,3,600,0.03,0.2892,10.3563
2,4,350,0.03,0.2888,8.3299
0,3,350,0.03,0.2860,9.0155



Best internal PR-AUC: 0.2925
Best XGBoost configuration: {'learning_rate': 0.05, 'min_child_weight': 10, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 5, 'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'aucpr', 'random_state': 42, 'n_jobs': 4, 'max_depth': 4, 'n_estimators': 600}



## 04.6 HistGradientBoosting tuning

HistGradientBoosting performed similarly to XGBoost in our initial
comparison. We now test whether changing the number of leaves,
boosting iterations and learning rate improves its performance.

We use the same internal chronological split as XGBoost and select
the best configuration using PR-AUC. The main validation set remains
reserved for comparing the tuned models.


In [13]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from IPython.display import display

import numpy as np
import pandas as pd

train_dates = pd.to_datetime(train_df["issue_date"])

fold = np.where(
    train_dates < pd.Timestamp("2015-04-01"),
    -1, 0
)

cv = PredefinedSplit(test_fold=fold)

hist_base = {
    "max_depth": 5,
    "min_samples_leaf": 50,
    "l2_regularization": 5,
    "early_stopping": False,
    "random_state": 42,
}

hist_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", HistGradientBoostingClassifier(**hist_base)),
])

param_grid = {
    "model__max_leaf_nodes": [15, 31],
    "model__max_iter": [250, 350],
    "model__learning_rate": [0.03, 0.05],
}

hist_search = GridSearchCV(
    estimator=hist_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    refit=False,
    n_jobs=1,
    verbose=0,
)

hist_search.fit(X_train, y_train)

hist_grid_results = (
    pd.DataFrame(hist_search.cv_results_)[
        [
            "param_model__max_leaf_nodes",
            "param_model__max_iter",
            "param_model__learning_rate",
            "mean_test_score",
            "mean_fit_time",
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
    .reset_index(drop=True)
)

display(hist_grid_results.round(4))

hist_best_config = {
    **hist_base,
    **{
        key.replace("model__", ""): value
        for key, value in hist_search.best_params_.items()
    },
}

print("Best internal PR-AUC:", round(hist_search.best_score_, 4))
print("Best configuration:", hist_best_config)


,param_model__max_leaf_nodes,param_model__max_iter,param_model__learning_rate,mean_test_score,mean_fit_time
0,15,350,0.05,0.2909,21.9145
1,31,350,0.05,0.2907,18.6782
2,15,250,0.05,0.2899,10.6712
3,31,250,0.05,0.2898,14.7568
4,31,350,0.03,0.2896,14.1247
5,15,350,0.03,0.2887,14.3840
6,31,250,0.03,0.2877,13.3753
7,15,250,0.03,0.2866,10.5331


Best internal PR-AUC: 0.2909
Best configuration: {'max_depth': 5, 'min_samples_leaf': 50, 'l2_regularization': 5, 'early_stopping': False, 'random_state': 42, 'learning_rate': 0.05, 'max_iter': 350, 'max_leaf_nodes': 15}



## 04.7 CatBoost tuning

CatBoost performed close to XGBoost and HistGradientBoosting in
the initial comparison. We now test different tree depths and
boosting iterations to see whether its performance improves.

CatBoost handles categorical features directly and can also
handle missing numerical values. We use the same chronological
split and PR-AUC selection criterion as the other models.

The best configuration will be compared with the other tuned
models on our late-2015 validation set.


In [14]:

from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from IPython.display import display

import numpy as np
import pandas as pd

train_dates = pd.to_datetime(
    train_df["issue_date"]
).reset_index(drop=True)

cv = PredefinedSplit(
    test_fold=np.where(
        train_dates < pd.Timestamp("2015-04-01"),
        -1, 0
    )
)

X_cat_search = X_train.copy()

for col in categorical_cols:
    X_cat_search[col] = (
        X_cat_search[col]
        .fillna("MISSING")
        .astype(str)
    )

cat_base = {
    "loss_function": "Logloss",
    "l2_leaf_reg": 5,
    "random_seed": 42,
    "thread_count": 4,
    "verbose": False,
    "allow_writing_files": False,
}

param_grid = {
    "depth": [4, 5, 6],
    "iterations": [350, 500],
    "learning_rate": [0.05],
}

cat_search = GridSearchCV(
    estimator=CatBoostClassifier(**cat_base),
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    refit=False,
    n_jobs=1,
    verbose=0,
)

cat_search.fit(
    X_cat_search,
    y_train,
    cat_features=categorical_cols
)

cat_grid_results = (
    pd.DataFrame(cat_search.cv_results_)[
        [
            "param_depth",
            "param_iterations",
            "param_learning_rate",
            "mean_test_score",
            "mean_fit_time",
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
    .reset_index(drop=True)
)

display(cat_grid_results.round(4))

cat_best_config = {
    **cat_base,
    **cat_search.best_params_,
}

print("\nBest internal PR-AUC:", round(cat_search.best_score_, 4))
print("Best configuration:", cat_best_config)


,param_depth,param_iterations,param_learning_rate,mean_test_score,mean_fit_time
0,6,500,0.05,0.2934,57.6516
1,6,350,0.05,0.2925,44.2696
2,5,500,0.05,0.2920,47.6111
3,4,500,0.05,0.2916,41.3629
4,5,350,0.05,0.2907,35.2282
5,4,350,0.05,0.2898,30.7037



Best internal PR-AUC: 0.2934
Best configuration: {'loss_function': 'Logloss', 'l2_leaf_reg': 5, 'random_seed': 42, 'thread_count': 4, 'verbose': False, 'allow_writing_files': False, 'depth': 6, 'iterations': 500, 'learning_rate': 0.05}



## 04.8 Comparing the tuned models

We selected the best configurations for XGBoost, HistGradientBoosting
and CatBoost using the internal chronological validation split.

We now retrain these configurations on the complete training period
and compare them with the original models on late-2015 validation data.

PR-AUC remains the main selection metric. We also examine ROC-AUC,
F1-score, calibration error and training time to see whether the
improvements justify the additional model complexity.


In [15]:

import time
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)
from IPython.display import display

# Use the encoded data prepared in 04.3
tuned_runs = {}

models = {
    "XGBoost": XGBClassifier(**xgb_best_config),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        **hist_best_config
    ),
}

for name, model in models.items():
    start = time.perf_counter()

    model.fit(X_train_encoded, y_train)
    prob = model.predict_proba(X_val_encoded)[:, 1]

    tuned_runs[name] = {
        "model": model,
        "prob": prob,
        "seconds": time.perf_counter() - start,
        "config": model.get_params(),
        "preprocessor": preprocessor,
    }

    print(f"{name}: {tuned_runs[name]['seconds']:.1f}s", flush=True)


# CatBoost uses raw numerical values and native categorical features,
# matching the preprocessing used in its grid search.
X_cat_train = X_train.copy()
X_cat_val = X_val.copy()

for col in categorical_cols:
    X_cat_train[col] = X_cat_train[col].fillna("MISSING").astype(str)
    X_cat_val[col] = X_cat_val[col].fillna("MISSING").astype(str)

cat_params = {
    k: v for k, v in cat_best_config.items()
    if k != "cat_features"
}

cat_tuned = CatBoostClassifier(**cat_params)

start = time.perf_counter()

cat_tuned.fit(
    X_cat_train,
    y_train,
    cat_features=categorical_cols,
)

tuned_runs["CatBoost"] = {
    "model": cat_tuned,
    "prob": cat_tuned.predict_proba(X_cat_val)[:, 1],
    "seconds": time.perf_counter() - start,
    "config": cat_params,
    "preprocessor": None,
}

print(
    f"CatBoost: {tuned_runs['CatBoost']['seconds']:.1f}s",
    flush=True,
)


def evaluate_run(name, run, version):
    prob = np.asarray(run["prob"]).ravel()

    precision, recall, thresholds = precision_recall_curve(
        y_val, prob
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )

    idx = int(np.argmax(f1_values))
    threshold = float(thresholds[idx])
    pred = (prob >= threshold).astype(int)

    return {
        "model": name,
        "version": version,
        "roc_auc": roc_auc_score(y_val, prob),
        "pr_auc": average_precision_score(y_val, prob),
        "best_f1": f1_score(y_val, pred),
        "threshold": threshold,
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "brier": brier_score_loss(y_val, prob),
        "train_seconds": run["seconds"],
    }


rows = []

for name in ["XGBoost", "HistGradientBoosting", "CatBoost"]:
    rows.append(evaluate_run(name, family_runs[name], "Baseline"))
    rows.append(evaluate_run(name, tuned_runs[name], "Tuned"))

comparison = pd.DataFrame(rows)

print("\nBASELINE VS TUNED")
display(
    comparison.sort_values(["model", "version"])
    .reset_index(drop=True)
    .round(4)
)

print("\nCHANGE IN PR-AUC AFTER TUNING")
improvement = comparison.pivot(
    index="model",
    columns="version",
    values="pr_auc",
)

improvement["improvement"] = (
    improvement["Tuned"] - improvement["Baseline"]
)

display(improvement.round(5))

print("\nBEST TUNED MODEL")
tuned_comparison = (
    comparison[comparison["version"] == "Tuned"]
    .sort_values("pr_auc", ascending=False)
)

display(tuned_comparison.round(4))


XGBoost: 14.7s
HistGradientBoosting: 19.3s
CatBoost: 90.9s

BASELINE VS TUNED


,model,version,roc_auc,pr_auc,best_f1,threshold,precision,recall,brier,train_seconds
0,CatBoost,Baseline,0.7128,0.2963,0.3630,0.1566,0.2723,0.5441,0.1176,67.5447
1,CatBoost,Tuned,0.7147,0.2980,0.3647,0.1595,0.2744,0.5438,0.1173,90.8709
2,HistGradientBoosting,Baseline,0.7136,0.2972,0.3641,0.1520,0.2656,0.5788,0.1173,24.4086
3,HistGradientBoosting,Tuned,0.7132,0.2970,0.3639,0.1543,0.2682,0.5661,0.1174,19.3339
4,XGBoost,Baseline,0.7133,0.2980,0.3633,0.1501,0.2645,0.5799,0.1174,9.0753
5,XGBoost,Tuned,0.7149,0.2991,0.3644,0.1561,0.2693,0.5637,0.1171,14.7321



CHANGE IN PR-AUC AFTER TUNING


version,Baseline,Tuned,improvement
model,,,
CatBoost,0.29634,0.29804,0.00171
HistGradientBoosting,0.29720,0.29703,-0.00017
XGBoost,0.29796,0.29912,0.00115



BEST TUNED MODEL


,model,version,roc_auc,pr_auc,best_f1,threshold,precision,recall,brier,train_seconds
1,XGBoost,Tuned,0.7149,0.2991,0.3644,0.1561,0.2693,0.5637,0.1171,14.7321
5,CatBoost,Tuned,0.7147,0.2980,0.3647,0.1595,0.2744,0.5438,0.1173,90.8709
3,HistGradientBoosting,Tuned,0.7132,0.2970,0.3639,0.1543,0.2682,0.5661,0.1174,19.3339



### Model evaluation

We compare the baseline and tuned models using ranking, classification
and probability metrics.

PR-AUC is our primary selection metric because we want to identify
defaults in an imbalanced dataset without committing to a threshold
too early. F1, precision and recall are evaluated at each model's
best validation threshold.

We also compare performance at the standard 0.5 threshold, risk
concentration among the highest-scored loans, calibration error
and training time.

The best validation F1 is exploratory, since its threshold is chosen
using the same validation data.


In [16]:

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)


def evaluate_run(name, run, version):
    prob = np.asarray(run["prob"]).ravel()
    y = np.asarray(y_val).ravel()

    precision, recall, thresholds = precision_recall_curve(y, prob)

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )

    threshold = float(thresholds[np.argmax(f1_values)])

    pred = (prob >= threshold).astype(int)
    pred_05 = (prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    top_n = int(np.ceil(len(prob) * 0.10))
    top_idx = np.argsort(-prob, kind="stable")[:top_n]

    top10_default_rate = y[top_idx].mean()
    top10_lift = top10_default_rate / y.mean()

    return {
        "model": name,
        "version": version,
        "pr_auc": average_precision_score(y, prob),
        "roc_auc": roc_auc_score(y, prob),
        "best_f1": f1_score(y, pred),
        "threshold": threshold,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "balanced_acc": balanced_accuracy_score(y, pred),
        "accuracy": accuracy_score(y, pred),
        "f1_at_0.5": f1_score(y, pred_05, zero_division=0),
        "brier": brier_score_loss(y, prob),
        "log_loss": log_loss(y, prob),
        "top10_default_rate": top10_default_rate,
        "top10_lift": top10_lift,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "train_seconds": run["seconds"],
    }


model_names = [
    "XGBoost",
    "HistGradientBoosting",
    "CatBoost",
]

rows = []

for name in model_names:
    rows.append(evaluate_run(name, family_runs[name], "Baseline"))
    rows.append(evaluate_run(name, tuned_runs[name], "Tuned"))

comparison = pd.DataFrame(rows)

print("BASELINE VS TUNED MODELS")

display(
    comparison[
        [
            "model", "version", "pr_auc", "roc_auc",
            "best_f1", "threshold", "precision",
            "recall", "balanced_acc", "accuracy",
            "f1_at_0.5", "brier", "log_loss",
            "top10_lift", "train_seconds",
        ]
    ].round(4)
)


# Improvement from hyperparameter tuning
metrics = [
    "pr_auc", "roc_auc", "best_f1",
    "brier", "top10_lift",
]

baseline_scores = (
    comparison[comparison["version"] == "Baseline"]
    .set_index("model")[metrics]
)

tuned_scores = (
    comparison[comparison["version"] == "Tuned"]
    .set_index("model")[metrics]
)

improvement = tuned_scores - baseline_scores
improvement.columns = [
    f"delta_{col}" for col in improvement.columns
]

print("\nIMPROVEMENT AFTER TUNING")
display(improvement.round(5))


# Rank all six models using validation PR-AUC
ranking = (
    comparison
    .sort_values(
        ["pr_auc", "roc_auc"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nALL MODELS RANKED BY PR-AUC")

display(
    ranking[
        [
            "model", "version",
            "pr_auc", "roc_auc", "best_f1",
            "threshold", "precision", "recall",
            "brier", "top10_lift", "train_seconds",
        ]
    ].round(4)
)


# Confusion matrices for the three tuned models
print("\nTUNED MODEL CONFUSION MATRICES")

display(
    comparison.loc[
        comparison["version"] == "Tuned",
        ["model", "TP", "FP", "FN", "TN"]
    ].reset_index(drop=True)
)


# Save one consolidated comparison
report_dir = root / "reports/modeling"
report_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    report_dir / "04_baseline_vs_tuned_comparison.csv",
    index=False,
)

print("\nHighest validation PR-AUC:")
print(ranking.iloc[0][["model", "version", "pr_auc"]])

print("\nComparison saved.")
print("2016 test set untouched.")


BASELINE VS TUNED MODELS


,model,version,pr_auc,roc_auc,best_f1,threshold,precision,recall,balanced_acc,accuracy,f1_at_0.5,brier,log_loss,top10_lift,train_seconds
0,XGBoost,Baseline,0.2980,0.7133,0.3633,0.1501,0.2645,0.5799,0.6494,0.6982,0.0074,0.1174,0.3844,2.4190,9.0753
1,XGBoost,Tuned,0.2991,0.7149,0.3644,0.1561,0.2693,0.5637,0.6485,0.7082,0.0117,0.1171,0.3834,2.4183,14.7321
2,HistGradientBoosting,Baseline,0.2972,0.7136,0.3641,0.1520,0.2656,0.5788,0.6499,0.6999,0.0083,0.1173,0.3841,2.3910,24.4086
3,HistGradientBoosting,Tuned,0.2970,0.7132,0.3639,0.1543,0.2682,0.5661,0.6484,0.7063,0.0063,0.1174,0.3843,2.3765,19.3339
4,CatBoost,Baseline,0.2963,0.7128,0.3630,0.1566,0.2723,0.5441,0.6453,0.7165,0.0053,0.1176,0.3850,2.3750,67.5447
5,CatBoost,Tuned,0.2980,0.7147,0.3647,0.1595,0.2744,0.5438,0.6465,0.7188,0.0096,0.1173,0.3839,2.4069,90.8709



IMPROVEMENT AFTER TUNING


,delta_pr_auc,delta_roc_auc,delta_best_f1,delta_brier,delta_top10_lift
model,,,,,
XGBoost,0.00115,0.00159,0.00116,-0.00028,-0.00076
HistGradientBoosting,-0.00017,-0.00035,-0.00013,0.00006,-0.01441
CatBoost,0.00171,0.00190,0.00172,-0.00031,0.03186



ALL MODELS RANKED BY PR-AUC


,model,version,pr_auc,roc_auc,best_f1,threshold,precision,recall,brier,top10_lift,train_seconds
0,XGBoost,Tuned,0.2991,0.7149,0.3644,0.1561,0.2693,0.5637,0.1171,2.4183,14.7321
1,CatBoost,Tuned,0.2980,0.7147,0.3647,0.1595,0.2744,0.5438,0.1173,2.4069,90.8709
2,XGBoost,Baseline,0.2980,0.7133,0.3633,0.1501,0.2645,0.5799,0.1174,2.4190,9.0753
3,HistGradientBoosting,Baseline,0.2972,0.7136,0.3641,0.1520,0.2656,0.5788,0.1173,2.3910,24.4086
4,HistGradientBoosting,Tuned,0.2970,0.7132,0.3639,0.1543,0.2682,0.5661,0.1174,2.3765,19.3339
5,CatBoost,Baseline,0.2963,0.7128,0.3630,0.1566,0.2723,0.5441,0.1176,2.3750,67.5447



TUNED MODEL CONFUSION MATRICES


,model,TP,FP,FN,TN
0,XGBoost,7431,20167,5752,55460
1,HistGradientBoosting,7463,20366,5720,55261
2,CatBoost,7169,18961,6014,56666



Highest validation PR-AUC:
model       XGBoost
version       Tuned
pr_auc     0.299118
Name: 0, dtype: object

Comparison saved.
2016 test set untouched.


In [17]:

import joblib
from pathlib import Path

model_dir = root / "saved_models"
model_dir.mkdir(parents=True, exist_ok=True)

selected = comparison.loc[
    (comparison["model"] == "XGBoost")
    & (comparison["version"] == "Tuned")
].iloc[0]

checkpoint = {
    "model": tuned_runs["XGBoost"]["model"],
    "preprocessor": tuned_runs["XGBoost"]["preprocessor"],
    "prepared_columns": list(X_train.columns),
    "numeric_features": numeric_features,
    "numeric_cols": numeric_cols,
    "indicator_cols": indicator_cols,
    "categorical_cols": categorical_cols,
    "common_purposes": sorted(common_purposes),
    "threshold": float(selected["threshold"]),
    "hyperparameters": xgb_best_config,
    "validation_pr_auc": float(selected["pr_auc"]),
}

path = model_dir / "04_xgb_tuned_checkpoint.pkl"
joblib.dump(checkpoint, path)

loaded = joblib.load(path)

assert loaded["prepared_columns"] == list(X_train.columns)

print("Checkpoint saved:", path)
print("PR-AUC:", round(loaded["validation_pr_auc"], 4))
print("Threshold:", round(loaded["threshold"], 4))


Checkpoint saved: d:\ML_Projects\P2P-Lending-Default-Prediction\saved_models\04_xgb_tuned_checkpoint.pkl
PR-AUC: 0.2991
Threshold: 0.1561



### Verifying the feature preparation module

The feature-cleaning code has been moved into `src/loanrisk/features.py`.

Before using it for training and deployment, we compare its outputs
with the features prepared earlier in this notebook. Both training
and validation data should match exactly, including column order,
data types, missing values and categorical values.

This ensures that the reusable implementation does not introduce
any changes to the selected model's inputs.


In [10]:

from pathlib import Path
import sys

import pandas as pd
from pandas.testing import assert_frame_equal

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from loanrisk.features import LoanFeatureBuilder

builder = LoanFeatureBuilder(numeric_features=list(numeric_features))

# Learn purpose categories using training data only
builder.fit(train_df)

X_train_new = builder.transform(train_df)
X_val_new = builder.transform(val_df)

assert_frame_equal(
    X_train_new.reset_index(drop=True),
    X_train.reset_index(drop=True),
    check_dtype=True,
    check_exact=True,
)

print("Training features: PASS")

assert_frame_equal(
    X_val_new.reset_index(drop=True),
    X_val.reset_index(drop=True),
    check_dtype=True,
    check_exact=True,
)

print("Validation features: PASS")

assert set(builder.common_purposes_) == set(common_purposes)

print("Purpose categories: PASS")
print("Feature order and data types: PASS")
print("Missing values: PASS")

print("\nPrepared columns:", X_train_new.shape[1])
print("Training shape:", X_train_new.shape)
print("Validation shape:", X_val_new.shape)
print("\nReusable feature preparation verified.")


Training features: PASS
Validation features: PASS
Purpose categories: PASS
Feature order and data types: PASS
Missing values: PASS

Prepared columns: 58
Training shape: (457355, 58)
Validation shape: (88810, 58)

Reusable feature preparation verified.



### Verifying the preprocessing module

We compare the reusable preprocessing implementation with the
preprocessor fitted during model optimization.

Both should produce the same encoded columns in the same order,
with identical values. This confirms that moving the preprocessing
logic into Python modules does not change model inputs.


In [11]:

from loanrisk.preprocessing import make_preprocessor
import numpy as np
import joblib
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

checkpoint = joblib.load(
    root / "saved_models/04_xgb_tuned_checkpoint.pkl"
)

original_pp = checkpoint["preprocessor"]

print("Original fitted preprocessor loaded successfully.")

# Rebuild and fit on the same training data
new_pp = make_preprocessor(numeric_features)
new_pp.fit(X_train)

# Compare a sample without creating another full encoded dataset
sample_train = X_train.iloc[:1000]
sample_val = X_val.iloc[:1000]

for name, sample in [
    ("Training", sample_train),
    ("Validation", sample_val),
]:
    original = original_pp.transform(sample).astype(np.float32)
    rebuilt = new_pp.transform(sample).astype(np.float32)

    np.testing.assert_allclose(
        original,
        rebuilt,
        rtol=0,
        atol=0,
    )

    print(f"{name} preprocessing: PASS")
    print("Encoded shape:", rebuilt.shape)

np.testing.assert_array_equal(
    original_pp.get_feature_names_out(),
    new_pp.get_feature_names_out(),
)

print("\nEncoded feature order: PASS")
print("Reusable preprocessing verified.")
print("2016 test set untouched.")


Original fitted preprocessor loaded successfully.
Training preprocessing: PASS
Encoded shape: (1000, 122)
Validation preprocessing: PASS
Encoded shape: (1000, 122)

Encoded feature order: PASS
Reusable preprocessing verified.
2016 test set untouched.



### Verifying `predict.py`

The reusable prediction module loads the saved XGBoost pipeline and accepts raw loan records containing all 55 selected features.

We verified that it detects missing columns, produces identical predictions to the saved pipeline, applies the selected threshold (0.1561), and preserves row order.

All checks passed. The 2016 test set remains untouched.


In [ ]:

from loanrisk.predict import load_model, predict_full
import numpy as np

bundle = load_model()

sample = val_df.iloc[:100].copy()

# Predictions through our new inference module
result = predict_full(sample, bundle)

# Predictions directly through the saved pipeline
expected = bundle["pipeline"].predict_proba(
    sample[bundle["raw_columns"]]
)[:, 1]

np.testing.assert_allclose(
    result["default_risk_score"].to_numpy(),
    expected,
    rtol=0,
    atol=0,
)

expected_flags = (
    expected >= bundle["threshold"]
).astype(int)

np.testing.assert_array_equal(
    result["predicted_default"].to_numpy(),
    expected_flags,
)

assert len(result) == 100
assert result.index.equals(sample.index)

# Missing-column check
try:
    predict_full(
        sample.drop(columns=[bundle["raw_columns"][0]]),
        bundle,
    )
except ValueError:
    print("Missing-column validation: PASS")
else:
    raise AssertionError("Missing-column validation failed.")

print("Prediction equivalence: PASS")
print("Classification threshold: PASS")
print("Row alignment: PASS")
print("Model:", bundle["model_name"])
print("2016 test set untouched.")

display(result.head())


Missing-column validation: PASS
Prediction equivalence: PASS
Classification threshold: PASS
Row alignment: PASS
Model: xgboost
2016 test set untouched.


,default_risk_score,predicted_default
0,0.097356,0
1,0.039031,0
2,0.031236,0
3,0.035207,0
4,0.015130,0
